 **-------------------------------------------------Sequence-to-Sequence Model (Mini Translator)----------------------------------------------------**

loading and cleaning the data

In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from sklearn.model_selection import train_test_split

# -----------------------------
# 1️⃣ Load and preprocess data
# -----------------------------
with open('urd.txt', 'r', encoding='utf-8') as file:
    text = file.read()

data = []
for line in text.split('\n'):
    if not line.strip():
        continue
    parts = line.split('\t')
    if len(parts) >= 2:
        english = parts[0].strip().lower()
        urdu = parts[1].strip()
        data.append([english, urdu])

df = pd.DataFrame(data, columns=['English', 'Urdu'])


Tokenizing the dataset

In [2]:
# -----------------------------
# 2️⃣ Tokenization
# -----------------------------

# English tokenizer
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(df['English'])
eng_seq = eng_tokenizer.texts_to_sequences(df['English'])

# Urdu tokenizer with startseq/endseq tokens
urdu_tokenizer = Tokenizer(filters='', lower=False)
urdu_with_tokens = ["startseq " + s + " endseq" for s in df['Urdu']]
urdu_tokenizer.fit_on_texts(urdu_with_tokens)
urdu_seq = urdu_tokenizer.texts_to_sequences(urdu_with_tokens)

Padding

In [3]:
# -----------------------------
# 3️⃣ Padding sequences
# -----------------------------
max_eng_len = max(len(s) for s in eng_seq)
max_urdu_len = max(len(s) for s in urdu_seq)

eng_seq_padded = pad_sequences(eng_seq, maxlen=max_eng_len, padding='post')
urdu_seq_padded = pad_sequences(urdu_seq, maxlen=max_urdu_len, padding='post')


Train test split and preparing input and output for decoder

In [4]:
# -----------------------------
# 4️⃣ Train/test split
# -----------------------------
eng_train, eng_val, urdu_train, urdu_val = train_test_split(
    eng_seq_padded, urdu_seq_padded, test_size=0.1, random_state=42
)

# Decoder input and target
urdu_seq_padded = np.array(urdu_seq_padded)
decoder_input_data = urdu_seq_padded[:, :-1]
decoder_target_data = urdu_seq_padded[:, 1:]


Encoder-Decoder Model Architecture

In [5]:
# -----------------------------
# 5️⃣ Build the model
# -----------------------------
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(len(eng_tokenizer.word_index)+1, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(len(urdu_tokenizer.word_index)+1, latent_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(len(urdu_tokenizer.word_index)+1, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Seq2seq model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │    339,712 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │    436,480 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    525,312 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    525,312 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │    438,185 │ lstm_1[0][0]      │
│                     │ 1705)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,265,001 (8.64 MB)

 Trainable params: 2,265,001 (8.64 MB)

 Non-trainable params: 0 (0.00 B)

Training the model

In [6]:
# -----------------------------
# 6️⃣ Train the model
# -----------------------------
history = model.fit(
    [eng_train, decoder_input_data[:len(eng_train)]],
    decoder_target_data[:len(eng_train), :, None],
    batch_size=32,
    epochs=20,
    validation_data=(
        [eng_val, decoder_input_data[len(eng_train):]],
        decoder_target_data[len(eng_train):, :, None]
    )
)


Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 15s 338ms/step - accuracy: 0.5679 - loss: 4.8933 - val_accuracy: 0.4289 - val_loss: 4.0650
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 317ms/step - accuracy: 0.6651 - loss: 2.0935 - val_accuracy: 0.4257 - val_loss: 3.8363
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 314ms/step - accuracy: 0.6907 - loss: 1.9473 - val_accuracy: 0.4652 - val_loss: 3.6537
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 307ms/step - accuracy: 0.7016 - loss: 1.8549 - val_accuracy: 0.4696 - val_loss: 3.6357
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 326ms/step - accuracy: 0.7037 - loss: 1.8027 - val_accuracy: 0.4708 - val_loss: 3.6213
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 317ms/step - accuracy: 0.7080 - loss: 1.7659 - val_accuracy: 0.4759 - val_loss: 3.6169
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 21s 320ms/step - accuracy: 0.7154 - loss: 1.6905 - val_accuracy: 0.4771 - val_loss: 3.6180
Epoch 8/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 20s 320ms/step - accuracy: 0.7151 - loss: 1.6960 - val_accu

Inference using trained model.

In [7]:
# -----------------------------
# 7️⃣ Inference setup
# -----------------------------
# Encoder inference model
encoder_model_inf = Model(encoder_inputs, encoder_states)

# Decoder inference model
decoder_inputs_inf = Input(shape=(1,))
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb_inf = dec_emb_layer(decoder_inputs_inf)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_emb_inf, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model_inf = Model(
    [decoder_inputs_inf] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

# -----------------------------
# 8️⃣ Decode function
# -----------------------------
def decode_sequence(input_seq):
    # Encode the input sentence to get initial states
    states_value = encoder_model_inf.predict(input_seq)

    # Start token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = urdu_tokenizer.word_index['startseq']

    decoded_sentence = []
    stop_condition = False

    while not stop_condition:
        output_tokens, h, c = decoder_model_inf.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = urdu_tokenizer.index_word.get(sampled_token_index, '')

        if sampled_word == 'endseq' or len(decoded_sentence) > max_urdu_len:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)

        # Update target_seq
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return ' '.join(decoded_sentence)



In [8]:
# -----------------------------
# 9️⃣ Test translation
# -----------------------------
from tensorflow.keras.preprocessing.sequence import pad_sequences

sample_input = "Is everything OK?"
input_seq = eng_tokenizer.texts_to_sequences([sample_input.lower()])
input_seq = pad_sequences(input_seq, maxlen=max_eng_len, padding='post')

translation = decode_sequence(input_seq)

print("Input:", sample_input)
print("Translation:", translation)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Input: Is everything OK?
Translation: میں نے اس نے اس کو لئیے سے لئیے ہوں۔
